# Adding a loop

## Loops in LangGraph

So far, we have seen workflows where the graph moves from one node to the next and eventually reaches an end point.

However, agentic applications often need to repeat steps. An agent may need to:
- check whether the current result is good enough,
- gather more information,
- try another approach,
- or refine its previous response.

This is where **loops** become useful.

In LangGraph, a loop is created by connecting a node back to an earlier point in the workflow. The agent can then continue running until a specific condition is met.

A simple example:

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv()

HF_TOKEN = os.getenv("HF_TOKEN")


print(r"""
   /\_/\\
  ( •ᴗ• )
  / >💜
HF token loaded
""")

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint

LLM = "huggingface"

def connect_to_llm(LLM):
    if LLM == "huggingface":
        llm = HuggingFaceEndpoint(
            repo_id="meta-llama/Llama-3.1-8B-Instruct",
            huggingfacehub_api_token=HF_TOKEN,
            task="text-generation",
            temperature=0.7, #control randomness (0= same answer every time, 0.7= more creative)
            max_new_tokens=256
        )
        return ChatHuggingFace(llm=llm)
    else:
        raise ValueError(f"Unsupported LLM: {LLM}")

In [ ]:
llm = connect_to_llm(LLM)

Adding a loop to our agent:

In [ ]:
from IPython.display import Image, display
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

# We will have a single message which is just a string
class AgentState(TypedDict):
    message: str

# In the main node, we compose the message and make decisions whether to add to it or to output the final text
def compose_greeting(state: AgentState) -> AgentState:
    print(f'Node 1: message currently says "{state["message"]}", deciding what to do next...')
    return state

# A functionality to add an additional exclamation mark, to be housed in its own node:
def add_exclamation_marks(state: AgentState) -> AgentState:
    print(f'Node 2: message updated to "{state["message"]}!"')
    return {"message": state["message"] + "!"}

# A function to decide whether to keep adding exclamation marks or to stop:
def should_continue(state: AgentState) -> str:
    if state["message"].count("!") >= 3:
        print("that's enough exclamation marks, we can stop now :) ")
        return "enough exclamation marks"
    return "less than 3 exclamation marks"

#Making a graph:
workflow = StateGraph(AgentState)

# Add main node to compose our text:
workflow.add_node("node1 compose greeting", compose_greeting)

# The node which has the functionality to add exclamation marks:
workflow.add_node("node2 add exclamation marks", add_exclamation_marks)

# from START we go to the main Node 1:
workflow.add_edge(START, "node1 compose greeting")

# from Node 1 we decide whether to go to Node 2 or to END, based on the number of exclamation marks:
workflow.add_conditional_edges(
    "node1 compose greeting",
    should_continue,
    {
        "less than 3 exclamation marks": "node2 add exclamation marks", #these are based on the return value of our decision function
        "enough exclamation marks": END,
    },
)
workflow.add_edge("node2 add exclamation marks", "node1 compose greeting") #from here we always go back to the main Node1

app = workflow.compile()
app

And now we can see how it works:

In [ ]:
app.invoke({'message': 'hi there'})